# 🏦 Credit Scoring Challenge — Альфа-Банк

Автоматизированный ноутбук для участия в соревновании по построению модели кредитного скоринга. Используется LightGBM, Optuna и полноценный пайплайн обработки.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

In [ ]:

from pathlib import Path

# Путь к загруженным данным
data_dir = Path(".'/Users/ivanzelenskiy/Desktop/Рабочий стол — MacBook Air — Ivan/Final_alfa_test'.")  # ЗАМЕНИТЕ на имя своей папки!

# Загрузка таргета
train_target = pd.read_csv(data_dir / "train_target.csv")

# Загрузка и объединение всех тренировочных данных
train_dfs = []
for i in range(12):
    df = pd.read_parquet(data_dir / f"train_data_{i}.parquet")
    train_dfs.append(df)
train_data = pd.concat(train_dfs, axis=0)

# Объединение с таргетом
train_df = train_data.merge(train_target, on="id")

print("Train shape:", train_df.shape)
train_df.head()


In [ ]:

X = train_df.drop(columns=["id", "rn", "target"])
y = train_df["target"]

# Обработка NaN
X = X.fillna(-999)

# Разделение
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Модель
model = LGBMClassifier(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42)

model.fit(X_train, y_train,
          eval_set=[(X_val, y_val)],
          early_stopping_rounds=50,
          verbose=100)

# Оценка
y_pred_val = model.predict_proba(X_val)[:, 1]
score = roc_auc_score(y_val, y_pred_val)
print(f"Validation ROC-AUC: {score:.4f}")


In [ ]:

# Загрузка тестовых данных
test_dfs = []
for i in range(2):
    df = pd.read_parquet(data_dir / f"test_data_{i}.parquet")
    test_dfs.append(df)
test_data = pd.concat(test_dfs, axis=0)

# ID и предобработка
test_ids = test_data["id"]
X_test = test_data.drop(columns=["id", "rn"])
X_test = X_test.fillna(-999)

# Предсказание
test_preds = model.predict_proba(X_test)[:, 1]

# Сохранение сабмита
submission = pd.DataFrame({"id": test_ids, "target": test_preds})
submission.to_csv("submission_final.csv", index=False)
submission.head()
